# ERA5-Land Climate Data Collection and Processing

This notebook retrieves and processes climate data from the Copernicus ERA5-Land dataset for Romanian counties.

The analysis focuses on two climate indicators:
- mean annual 2-metre temperature (°C);
- annual total precipitation (mm).

Monthly ERA5-Land data are aggregated temporally to annual values and spatially to the NUTS 3 county level. Bucharest is excluded to maintain consistency with the main analytical panel.

**Source:** Copernicus Climate Data Store (ERA5-Land)  
**Spatial level:** Romanian NUTS 3 counties  
**Final analysis period:** 2012–2022

In [11]:
import cdsapi
import xarray as xr
import pandas as pd
import geopandas as gpd
import rioxarray
import rasterio
from rasterstats import zonal_stats

print("Pachetele au fost importate cu succes.")

Pachetele au fost importate cu succes.


In [12]:
from pathlib import Path
from dotenv import load_dotenv
import os
import cdsapi

# Load credentials from the .env file in the repository root
env_path = Path("../../.env")
load_dotenv(env_path)

cds_url = os.getenv("CDS_BASE_URL")
cds_key = os.getenv("CDS_API_KEY")

if not cds_url or not cds_key:
    raise ValueError(
        "CDS credentials not found. Check CDS_BASE_URL and CDS_API_KEY in .env."
    )

client = cdsapi.Client(
    url=cds_url,
    key=cds_key
)

print("CDS client initialized successfully.")

CDS client initialized successfully.


### 1. ERA5-Land Data Retrieval

Monthly averaged ERA5-Land climate data were retrieved for Romania for the 2012–2024 period.

The downloaded NetCDF file includes:
- 2-metre temperature
- total precipitation
- all months between 2012 and 2024
- a spatial bounding box covering Romania and a small surrounding buffer

**Dataset:** `reanalysis-era5-land-monthly-means`

ERA5-Land is provided on a regular grid of approximately 0.1° × 0.1°, allowing the climate variables to be subsequently aggregated to the county level.

### ERA5-Land Monthly Data API Request

In [13]:
from pathlib import Path

# Folder inside the repository where raw climate data will be stored
climate_dir = Path("../../data/raw/climate")
climate_dir.mkdir(parents=True, exist_ok=True)

# Output NetCDF file
output_file = climate_dir / "era5_land_romania_2011_2024.nc"

dataset = "reanalysis-era5-land-monthly-means"

request = {
    "product_type": ["monthly_averaged_reanalysis"],

    "variable": [
        "2m_temperature",
        "total_precipitation"
    ],

    "year": [
        "2011", "2012", "2013", "2014", "2015", "2016", "2017",
        "2018", "2019", "2020", "2021", "2022", "2023", "2024"
    ],

    "month": [
        "01", "02", "03", "04", "05", "06",
        "07", "08", "09", "10", "11", "12"
    ],

    "time": ["00:00"],

    # Romania bounding box: North, West, South, East
    "area": [49, 20, 43, 30],

    "data_format": "netcdf",
    "download_format": "unarchived"
}

client.retrieve(
    dataset,
    request
).download(str(output_file))

print(f"Download completed: {output_file}")

2026-08-22 18:08:28,158 INFO Request ID is 56a77f32-cfa8-471e-9006-6e17c793f3c9
2026-08-22 18:08:28,281 INFO status has been updated to accepted
2026-08-22 18:08:42,281 INFO status has been updated to running
2026-08-22 18:09:45,066 INFO status has been updated to successful
                                                                                         

Download completed: ..\..\data\raw\climate\era5_land_romania_2011_2024.nc


In [14]:
import xarray as xr

ds = xr.open_dataset(output_file)
ds

<xarray.Dataset> Size: 8MB
Dimensions:     (valid_time: 168, latitude: 61, longitude: 101)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 1kB 2011-01-01 ... 2024-12-01
    expver      (valid_time) <U4 3kB ...
  * latitude    (latitude) float64 488B 49.0 48.9 48.8 48.7 ... 43.2 43.1 43.0
  * longitude   (longitude) float64 808B 20.0 20.1 20.2 20.3 ... 29.8 29.9 30.0
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 4MB ...
    tp          (valid_time, latitude, longitude) float32 4MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-22T15:09 GRIB to CDM+CF via cfgrib-0.9.1...

### 2. County-Level Spatial Data

Romanian county boundaries were obtained from the Eurostat GISCO NUTS 2021 dataset at NUTS 3 level.

**Spatial dataset:** `NUTS_RG_01M_2021_4326`

The county geometries are used to spatially aggregate the gridded ERA5-Land climate data to the county level.

In [23]:
import geopandas as gpd
from pathlib import Path

shp_path = Path("../../data/spatial/NUTS/NUTS_RG_01M_2021_4326.shp")

nuts = gpd.read_file(shp_path)

print(nuts.shape)
nuts.head()

(2010, 9)


,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,geometry
0,AL,0,AL,Shqipëria,Shqipëria,0.0,0,0,"MULTIPOLYGON (((19.75628 42.63384, 19.75005 42..."
1,CZ,0,CZ,Česko,Česko,0.0,0,0,"POLYGON ((14.33499 51.04007, 14.34494 51.03908..."
2,DE,0,DE,Deutschland,Deutschland,0.0,0,0,"MULTIPOLYGON (((10.45444 47.5558, 10.43954 47...."
3,DK,0,DK,Danmark,Danmark,0.0,0,0,"MULTIPOLYGON (((15.19309 55.32015, 15.19056 55..."
4,CY,0,CY,Kýpros,Κύπρος,0.0,0,0,"MULTIPOLYGON (((34.60609 35.70767, 34.6006 35...."


In [24]:
nuts.columns

Index(['NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NAME_LATN', 'NUTS_NAME',
       'MOUNT_TYPE', 'URBN_TYPE', 'COAST_TYPE', 'geometry'],
      dtype='object')

In [25]:
#filtrez Romania la nivel NUTS 3
judete_ro = nuts[
    (nuts["CNTR_CODE"] == "RO") &
    (nuts["LEVL_CODE"] == 3)
].copy()

print(judete_ro.shape)
judete_ro[["NUTS_ID", "NAME_LATN", "CNTR_CODE", "LEVL_CODE"]].head()

(42, 9)


,NUTS_ID,NAME_LATN,CNTR_CODE,LEVL_CODE
1343,RO122,Braşov,RO,3
1349,RO424,Timiş,RO,3
1368,RO317,Teleorman,RO,3
1387,RO221,Brăila,RO,3
1388,RO316,Prahova,RO,3


In [26]:
judete_ro[["NUTS_ID", "NAME_LATN"]].sort_values("NAME_LATN")

,NUTS_ID,NAME_LATN
1798,RO121,Alba
1867,RO421,Arad
1668,RO311,Argeş
1643,RO211,Bacău
1664,RO111,Bihor
1596,RO112,Bistriţa-Năsăud
1666,RO212,Botoşani
1343,RO122,Braşov
1387,RO221,Brăila
1398,RO321,Bucureşti


In [27]:
#eliminam bucuresti

judete_ro_fara_buc = judete_ro[
    ~judete_ro["NAME_LATN"].str.contains("Bucure", case=False, na=False)
].copy()

print(judete_ro_fara_buc.shape)
judete_ro_fara_buc[["NUTS_ID", "NAME_LATN"]].sort_values("NAME_LATN")

(41, 9)


,NUTS_ID,NAME_LATN
1798,RO121,Alba
1867,RO421,Arad
1668,RO311,Argeş
1643,RO211,Bacău
1664,RO111,Bihor
1596,RO112,Bistriţa-Năsăud
1666,RO212,Botoşani
1343,RO122,Braşov
1387,RO221,Brăila
1855,RO222,Buzău


In [28]:
#verific sistem de coordonate 

print(judete_ro_fara_buc.crs)

EPSG:4326


In [29]:
#creez coloana judet 

judete_ro_fara_buc["judet"] = judete_ro_fara_buc["NAME_LATN"]

judete_ro_fara_buc[["NUTS_ID", "judet"]].sort_values("judet").head()

,NUTS_ID,judet
1798,RO121,Alba
1867,RO421,Arad
1668,RO311,Argeş
1643,RO211,Bacău
1664,RO111,Bihor


### 3. ERA5-Land Climate Data Processing

In [30]:
import xarray as xr
from pathlib import Path

nc_path = Path("../../data/raw/climate/era5_land_romania_2011_2024.nc")

ds = xr.open_dataset(nc_path)

ds

<xarray.Dataset> Size: 8MB
Dimensions:     (valid_time: 168, latitude: 61, longitude: 101)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 1kB 2011-01-01 ... 2024-12-01
    expver      (valid_time) <U4 3kB ...
  * latitude    (latitude) float64 488B 49.0 48.9 48.8 48.7 ... 43.2 43.1 43.0
  * longitude   (longitude) float64 808B 20.0 20.1 20.2 20.3 ... 29.8 29.9 30.0
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 4MB ...
    tp          (valid_time, latitude, longitude) float32 4MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-22T15:09 GRIB to CDM+CF via cfgrib-0.9.1...

In [31]:
#nume variabile & dimensiuni 

print("Variabile:", list(ds.data_vars))
print("Coordonate:", list(ds.coords))
print("Dimensiuni:", ds.dims)

Variabile: ['t2m', 'tp']
Coordonate: ['number', 'valid_time', 'latitude', 'longitude', 'expver']
Dimensiuni: FrozenMappingWarningOnValuesAccess({'valid_time': 168, 'latitude': 61, 'longitude': 101})


In [32]:
#verific perioada temporala 

print(ds["valid_time"].min().values)
print(ds["valid_time"].max().values)
print(len(ds["valid_time"]))

2011-01-01T00:00:00.000000000
2024-12-01T00:00:00.000000000
168


In [33]:
print(judete_ro_fara_buc.shape)
print(judete_ro_fara_buc.crs)

judete_ro_fara_buc[["NUTS_ID", "judet", "NAME_LATN"]].sort_values("judet").head()

(41, 10)
EPSG:4326


,NUTS_ID,judet,NAME_LATN
1798,RO121,Alba,Alba
1867,RO421,Arad,Arad
1668,RO311,Argeş,Argeş
1643,RO211,Bacău,Bacău
1664,RO111,Bihor,Bihor


In [34]:
#copie de lucru pt judete 

judete = judete_ro_fara_buc.copy()

judete = judete[["NUTS_ID", "judet", "geometry"]].copy()

judete.head()

,NUTS_ID,judet,geometry
1343,RO122,Braşov,"POLYGON ((25.26689 46.17611, 25.27272 46.17127..."
1349,RO424,Timiş,"POLYGON ((20.60895 46.14619, 20.61536 46.13992..."
1368,RO317,Teleorman,"POLYGON ((25.44462 44.46683, 25.46331 44.45907..."
1387,RO221,Brăila,"POLYGON ((27.57751 45.49113, 27.58836 45.48567..."
1388,RO316,Prahova,"POLYGON ((25.9387 45.51289, 25.9676 45.50847, ..."


In [35]:
print("Lat min:", float(ds.latitude.min()))
print("Lat max:", float(ds.latitude.max()))
print("Lon min:", float(ds.longitude.min()))
print("Lon max:", float(ds.longitude.max()))

print("Bounds județe:")
print(judete.total_bounds)

Lat min: 43.0
Lat max: 49.0
Lon min: 20.0
Lon max: 30.0
Bounds județe:
[20.264296   43.61962158 29.71262986 48.2644845 ]


#### Temperature and Precipitation Conversion

The ERA5-Land variables are converted into units suitable for the analysis:

- `t2m`: 2-metre temperature, provided in Kelvin
- `tp`: total precipitation, provided in metres

Temperature is converted from Kelvin to degrees Celsius:

`temperature_C = t2m - 273.15`

Monthly precipitation is converted to millimetres by accounting for the number of days in each month:

`monthly_precipitation_mm = tp × 1000 × number of days in month`

In [36]:
import pandas as pd
import xarray as xr

# copie de lucru
ds_work = ds.copy()

# temperatura: Kelvin -> Celsius
ds_work["t2m_c"] = ds_work["t2m"] - 273.15

# numărul de zile din fiecare lună
days_in_month = ds_work["valid_time"].dt.days_in_month

# precipitații: metri -> mm și ajustare la total lunar
ds_work["tp_mm_monthly"] = ds_work["tp"] * 1000 * days_in_month

ds_work[["t2m_c", "tp_mm_monthly"]]

<xarray.Dataset> Size: 12MB
Dimensions:        (valid_time: 168, latitude: 61, longitude: 101)
Coordinates:
  * valid_time     (valid_time) datetime64[ns] 1kB 2011-01-01 ... 2024-12-01
    expver         (valid_time) <U4 3kB ...
  * latitude       (latitude) float64 488B 49.0 48.9 48.8 ... 43.2 43.1 43.0
  * longitude      (longitude) float64 808B 20.0 20.1 20.2 ... 29.8 29.9 30.0
    number         int64 8B ...
Data variables:
    t2m_c          (valid_time, latitude, longitude) float32 4MB -5.023 ... nan
    tp_mm_monthly  (valid_time, latitude, longitude) float64 8MB 24.1 ... nan
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-22T15:09 GRIB to CDM+CF via cfgrib-0.9.1...

In [37]:
print("Temperatura C min:", float(ds_work["t2m_c"].min()))
print("Temperatura C max:", float(ds_work["t2m_c"].max()))

print("Precipitații lunare mm min:", float(ds_work["tp_mm_monthly"].min()))
print("Precipitații lunare mm max:", float(ds_work["tp_mm_monthly"].max()))

Temperatura C min: -12.165130615234375
Temperatura C max: 29.404693603515625
Precipitații lunare mm min: 0.19578633829951286
Precipitații lunare mm max: 343.9236831665039


#### Monthly-to-Annual Climate Aggregation

Monthly climate indicators are aggregated to annual values:

- **Annual mean temperature (°C):** arithmetic mean of the 12 monthly temperature values.
- **Annual precipitation (mm):** sum of the 12 monthly precipitation totals.

In [38]:
# Agregare anuală
temp_anuala = ds_work["t2m_c"].groupby("valid_time.year").mean("valid_time")

prec_anuala = ds_work["tp_mm_monthly"].groupby("valid_time.year").sum("valid_time")

print(temp_anuala)
print(prec_anuala)

<xarray.DataArray 't2m_c' (year: 14, latitude: 61, longitude: 101)> Size: 345kB
array([[[ 5.673838 ,  5.589854 ,  5.99057  , ...,  8.699554 ,
          8.735036 ,  8.743175 ],
        [ 5.208018 ,  5.241221 ,  5.496104 , ...,  8.768239 ,
          8.743337 ,  8.7465925],
        [ 5.8632913,  5.8279724,  6.1539817, ...,  8.838226 ,
          8.80486  ,  8.85613  ],
        ...,
        [ 7.0364685,  7.2198997,  7.3756614, ...,        nan,
                nan,        nan],
        [ 6.837087 ,  6.6966248,  7.390798 , ...,        nan,
                nan,        nan],
        [ 7.0582786,  6.5148215,  6.8727317, ...,        nan,
                nan,        nan]],

       [[ 5.250315 ,  5.1760964,  5.5968323, ...,  9.222646 ,
          9.246735 ,  9.256012 ],
        [ 4.788076 ,  4.819163 ,  5.091461 , ...,  9.317047 ,
          9.304515 ,  9.311839 ],
        [ 5.519358 ,  5.4936423,  5.8190002, ...,  9.424469 ,
          9.404775 ,  9.435211 ],
...
        [ 8.054799 ,  8.298289 ,  8.5

In [39]:
print("Temperatura anuală C min:", float(temp_anuala.min()))
print("Temperatura anuală C max:", float(temp_anuala.max()))

print("Precipitații anuale mm min:", float(prec_anuala.min()))
print("Precipitații anuale mm max:", float(prec_anuala.max()))

Temperatura anuală C min: 3.1449687480926514
Temperatura anuală C max: 15.69599437713623
Precipitații anuale mm min: 0.0
Precipitații anuale mm max: 1749.7316545248032


In [40]:
# găsește unde precipitația anuală este 0
zero_prec = prec_anuala.where(prec_anuala == 0, drop=True)

zero_prec

<xarray.DataArray 'tp_mm_monthly' (year: 14, latitude: 29, longitude: 22)> Size: 71kB
array([[[nan, nan, nan, ..., nan, nan,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
        [nan, nan,  0., ...,  0.,  0.,  0.],
        [nan,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]],

       [[nan, nan, nan, ..., nan, nan,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
        [nan, nan,  0., ...,  0.,  0.,  0.],
        [nan,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]],

       [[nan, nan, nan, ..., nan, nan,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
...
        ...,
        [nan, nan,  0., ...,  0.,  0.,  0.],
        [nan,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]],

       [[nan, nan, nan, ..., nan, nan,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
        [nan, nan,  0., ...,  0.,  0.,  0.],
        [nan,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]],

       [[nan, nan, nan, ..., nan, nan,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
        [nan, nan,  0., ...,  0.,  0.,  0.],
        [nan,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]]], shape=(14, 29, 22))
Coordinates:
  * year       (year) int64 112B 2011 2012 2013 2014 ... 2021 2022 2023 2024
  * latitude   (latitude) float64 232B 45.8 45.7 45.6 45.5 ... 43.2 43.1 43.0
  * longitude  (longitude) float64 176B 27.9 28.0 28.1 28.2 ... 29.8 29.9 30.0
    number     int64 8B 0
Attributes: (12/29)
    GRIB_paramId:                             228
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      6161
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_gridType:                            regular_ll
    ...                                       ...
    GRIB_name:                                Total precipitation
    GRIB_shortName:                           tp
    GRIB_totalNumber:                         0
    GRIB_units:                               m
    units:                                    m
    GRIB_surface:                             0.0

In [41]:
print("Număr valori anuale cu precipitații 0:", zero_prec.count().values)

Număr valori anuale cu precipitații 0: 4550


In [42]:
zero_df = zero_prec.to_dataframe(name="precipitatii_anuale_mm").reset_index()
zero_df = zero_df.dropna()

zero_df.head(20)

,year,latitude,longitude,number,precipitatii_anuale_mm
21,2011,45.8,30.0,0,0.0
39,2011,45.7,29.6,0,0.0
41,2011,45.7,29.8,0,0.0
42,2011,45.7,29.9,0,0.0
43,2011,45.7,30.0,0,0.0
62,2011,45.6,29.7,0,0.0
63,2011,45.6,29.8,0,0.0
64,2011,45.6,29.9,0,0.0
65,2011,45.6,30.0,0,0.0
84,2011,45.5,29.7,0,0.0


#### Spatial Coverage Validation

The spatial overlap between the ERA5-Land grid and Romanian NUTS 3 county boundaries was checked before county-level aggregation. Grid cells with zero values were also inspected to verify whether they fall within the Romanian county geometries.

In [44]:
import geopandas as gpd

# transformăm punctele cu precipitații 0 în GeoDataFrame
zero_gdf = gpd.GeoDataFrame(
    zero_df,
    geometry=gpd.points_from_xy(zero_df["longitude"], zero_df["latitude"]),
    crs="EPSG:4326"
)

# spatial join: vedem ce puncte zero cad în interiorul județelor
zero_in_judete = gpd.sjoin(
    zero_gdf,
    judete[["judet", "geometry"]],
    how="inner",
    predicate="within"
)

print("Număr puncte zero care cad în județele României:", len(zero_in_judete))

zero_in_judete[["year", "latitude", "longitude", "judet", "precipitatii_anuale_mm"]].head(50)

Număr puncte zero care cad în județele României: 168


,year,latitude,longitude,judet,precipitatii_anuale_mm
187,2011,45.0,29.0,Tulcea,0.0
209,2011,44.9,29.0,Tulcea,0.0
230,2011,44.8,28.9,Tulcea,0.0
231,2011,44.8,29.0,Tulcea,0.0
232,2011,44.8,29.1,Tulcea,0.0
251,2011,44.7,28.8,Tulcea,0.0
252,2011,44.7,28.9,Tulcea,0.0
253,2011,44.7,29.0,Tulcea,0.0
273,2011,44.6,28.8,Constanţa,0.0
274,2011,44.6,28.9,Constanţa,0.0


#### Treatment of Implausible Zero Precipitation Values

During climate data validation, annual precipitation values equal to 0 mm were identified in grid cells located in coastal counties, particularly Tulcea and Constanța.

Since an annual precipitation total of 0 mm is not plausible for Romania, and these observations were likely associated with water-covered or boundary grid cells, zero precipitation values were treated as missing before spatial aggregation to the county level.

In [45]:
print(prec_anuala)

<xarray.DataArray 'tp_mm_monthly' (year: 14, latitude: 61, longitude: 101)> Size: 690kB
array([[[ 872.74744639,  846.99661335,  810.90776853, ...,
          519.93658462,  515.99960485,  513.80550608],
        [ 901.63542035,  862.16860327,  814.4761183 , ...,
          525.48496529,  523.91410729,  522.22811118],
        [ 915.18933591,  871.81093076,  814.25214621, ...,
          531.46908143,  531.5540624 ,  530.9301956 ],
        ...,
        [ 704.63324732,  740.775271  ,  765.43452269, ...,
            0.        ,    0.        ,    0.        ],
        [ 668.51035923,  701.21360999,  733.88730019, ...,
            0.        ,    0.        ,    0.        ],
        [ 747.96761984,  784.43162006,  819.94997734, ...,
            0.        ,    0.        ,    0.        ]],

       [[ 983.39598715,  954.05538988,  903.20039821, ...,
          570.68884283,  568.48432046,  566.69104934],
        [1021.98065543,  975.2951889 ,  915.71179104, ...,
          560.01492494,  561.38732553,  

In [46]:
prec_anuala = prec_anuala.rename("precipitatii_anuale_mm")

In [47]:
prec_anuala_clean = prec_anuala.where(prec_anuala > 0)
prec_anuala_clean = prec_anuala_clean.rename("precipitatii_anuale_mm")

In [48]:
print("Min după curățare:", float(prec_anuala_clean.min()))
print("Max după curățare:", float(prec_anuala_clean.max()))

Min după curățare: 212.52328726649284
Max după curățare: 1749.7316545248032


In [49]:
temp_anuala = temp_anuala.rename("temperatura_medie_anuala_C")


### 4. County-Level Spatial Aggregation

In [50]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from rasterstats import zonal_stats
import tempfile
import rioxarray

In [51]:
judete = judete_ro_fara_buc.copy()

# păstrăm doar coloanele necesare
judete = judete[["NUTS_ID", "judet", "geometry"]].copy()

# ne asigurăm că CRS-ul este EPSG:4326
judete = judete.to_crs("EPSG:4326")

# reparare geometrie
judete["geometry"] = judete["geometry"].buffer(0)

print(judete.shape)
print(judete.crs)
judete.head(5)

(41, 3)
EPSG:4326


,NUTS_ID,judet,geometry
1343,RO122,Braşov,"POLYGON ((25.26689 46.17611, 25.27272 46.17127..."
1349,RO424,Timiş,"POLYGON ((20.60895 46.14619, 20.61536 46.13992..."
1368,RO317,Teleorman,"POLYGON ((25.44462 44.46683, 25.46331 44.45907..."
1387,RO221,Brăila,"POLYGON ((27.57751 45.49113, 27.58836 45.48567..."
1388,RO316,Prahova,"POLYGON ((25.9387 45.51289, 25.9676 45.50847, ..."


In [52]:
temp_anuala = temp_anuala.rename("temperatura_medie_anuala_C")

prec_anuala_clean = prec_anuala_clean.rename("precipitatii_anuale_mm")

In [54]:
# Check that temp_anuala and prec_anuala have the same dimensions (year, latitude, longitude)
print(temp_anuala)
print(prec_anuala_clean)

<xarray.DataArray 'temperatura_medie_anuala_C' (year: 14, latitude: 61,
                                                longitude: 101)> Size: 345kB
array([[[ 5.673838 ,  5.589854 ,  5.99057  , ...,  8.699554 ,
          8.735036 ,  8.743175 ],
        [ 5.208018 ,  5.241221 ,  5.496104 , ...,  8.768239 ,
          8.743337 ,  8.7465925],
        [ 5.8632913,  5.8279724,  6.1539817, ...,  8.838226 ,
          8.80486  ,  8.85613  ],
        ...,
        [ 7.0364685,  7.2198997,  7.3756614, ...,        nan,
                nan,        nan],
        [ 6.837087 ,  6.6966248,  7.390798 , ...,        nan,
                nan,        nan],
        [ 7.0582786,  6.5148215,  6.8727317, ...,        nan,
                nan,        nan]],

       [[ 5.250315 ,  5.1760964,  5.5968323, ...,  9.222646 ,
          9.246735 ,  9.256012 ],
        [ 4.788076 ,  4.819163 ,  5.091461 , ...,  9.317047 ,
          9.304515 ,  9.311839 ],
        [ 5.519358 ,  5.4936423,  5.8190002, ...,  9.424469 ,
      

In [55]:
# Function for county-level spatial aggregation

def zonal_mean_by_county(data_array, var_name, judete_gdf):
    """
    Calculează media spațială a unei variabile climatice grilate
    pentru fiecare județ și an.
    
    data_array trebuie să aibă dimensiunile:
    year, latitude, longitude
    """
    
    results = []
    
    temp_dir = Path(tempfile.mkdtemp())
    print("Folder temporar:", temp_dir)
    
    for year in data_array["year"].values:
        print(f"Procesez anul {year} pentru {var_name}...")
        
        da_year = data_array.sel(year=year)
        
        # setare dimensiuni spațiale pentru rioxarray
        da_year = da_year.rio.set_spatial_dims(
            x_dim="longitude",
            y_dim="latitude"
        )
        
        da_year = da_year.rio.write_crs("EPSG:4326")
        
        # setăm NaN ca nodata
        da_year = da_year.rio.write_nodata(np.nan)
        
        raster_path = temp_dir / f"{var_name}_{int(year)}.tif"
        da_year.rio.to_raster(raster_path)
        
        stats = zonal_stats(
            judete_gdf,
            raster_path,
            stats=["mean"],
            all_touched=True,
            nodata=np.nan
        )
        
        for idx, stat in enumerate(stats):
            results.append({
                "judet": judete_gdf.iloc[idx]["judet"],
                "NUTS_ID": judete_gdf.iloc[idx]["NUTS_ID"],
                "an": int(year),
                var_name: stat["mean"]
            })
    
    return pd.DataFrame(results)

In [56]:
# Aggregate temperature data by county

df_temp_judete = zonal_mean_by_county(
    data_array=temp_anuala,
    var_name="temperatura_medie_anuala_C",
    judete_gdf=judete
)

df_temp_judete.head()

Folder temporar: C:\Users\ASUS\AppData\Local\Temp\tmpomrsn493
Procesez anul 2011 pentru temperatura_medie_anuala_C...
Procesez anul 2012 pentru temperatura_medie_anuala_C...
Procesez anul 2013 pentru temperatura_medie_anuala_C...
Procesez anul 2014 pentru temperatura_medie_anuala_C...
Procesez anul 2015 pentru temperatura_medie_anuala_C...
Procesez anul 2016 pentru temperatura_medie_anuala_C...
Procesez anul 2017 pentru temperatura_medie_anuala_C...
Procesez anul 2018 pentru temperatura_medie_anuala_C...
Procesez anul 2019 pentru temperatura_medie_anuala_C...
Procesez anul 2020 pentru temperatura_medie_anuala_C...
Procesez anul 2021 pentru temperatura_medie_anuala_C...
Procesez anul 2022 pentru temperatura_medie_anuala_C...
Procesez anul 2023 pentru temperatura_medie_anuala_C...
Procesez anul 2024 pentru temperatura_medie_anuala_C...


,judet,NUTS_ID,an,temperatura_medie_anuala_C
0,Braşov,RO122,2011,7.111858
1,Timiş,RO424,2011,11.424029
2,Teleorman,RO317,2011,11.723287
3,Brăila,RO221,2011,11.461224
4,Prahova,RO316,2011,9.001926


In [57]:
print(df_temp_judete.shape)
df_temp_judete.head()

(574, 4)


,judet,NUTS_ID,an,temperatura_medie_anuala_C
0,Braşov,RO122,2011,7.111858
1,Timiş,RO424,2011,11.424029
2,Teleorman,RO317,2011,11.723287
3,Brăila,RO221,2011,11.461224
4,Prahova,RO316,2011,9.001926


In [59]:
# Aggregate precipitation data by county

df_prec_judete = zonal_mean_by_county(
    data_array=prec_anuala_clean,
    var_name="precipitatii_anuale_mm",
    judete_gdf=judete
)

df_prec_judete.head()

Folder temporar: C:\Users\ASUS\AppData\Local\Temp\tmp9kjan2z6
Procesez anul 2011 pentru precipitatii_anuale_mm...
Procesez anul 2012 pentru precipitatii_anuale_mm...
Procesez anul 2013 pentru precipitatii_anuale_mm...
Procesez anul 2014 pentru precipitatii_anuale_mm...
Procesez anul 2015 pentru precipitatii_anuale_mm...
Procesez anul 2016 pentru precipitatii_anuale_mm...
Procesez anul 2017 pentru precipitatii_anuale_mm...
Procesez anul 2018 pentru precipitatii_anuale_mm...
Procesez anul 2019 pentru precipitatii_anuale_mm...
Procesez anul 2020 pentru precipitatii_anuale_mm...
Procesez anul 2021 pentru precipitatii_anuale_mm...
Procesez anul 2022 pentru precipitatii_anuale_mm...
Procesez anul 2023 pentru precipitatii_anuale_mm...
Procesez anul 2024 pentru precipitatii_anuale_mm...


,judet,NUTS_ID,an,precipitatii_anuale_mm
0,Braşov,RO122,2011,742.395851
1,Timiş,RO424,2011,521.193295
2,Teleorman,RO317,2011,508.919199
3,Brăila,RO221,2011,447.970358
4,Prahova,RO316,2011,652.893368


In [60]:
print(df_prec_judete.shape)
df_prec_judete.head()

(574, 4)


,judet,NUTS_ID,an,precipitatii_anuale_mm
0,Braşov,RO122,2011,742.395851
1,Timiş,RO424,2011,521.193295
2,Teleorman,RO317,2011,508.919199
3,Brăila,RO221,2011,447.970358
4,Prahova,RO316,2011,652.893368


In [61]:
# Merge temperature and precipitation data

df_clima = df_temp_judete.merge(
    df_prec_judete[["judet", "an", "precipitatii_anuale_mm"]],
    on=["judet", "an"],
    how="inner"
)

print(df_clima.shape)
df_clima.head()

(574, 5)


,judet,NUTS_ID,an,temperatura_medie_anuala_C,precipitatii_anuale_mm
0,Braşov,RO122,2011,7.111858,742.395851
1,Timiş,RO424,2011,11.424029,521.193295
2,Teleorman,RO317,2011,11.723287,508.919199
3,Brăila,RO221,2011,11.461224,447.970358
4,Prahova,RO316,2011,9.001926,652.893368


#### Interpretation of County-Level Climate Indicators

For temperature, the annual values are spatially averaged across the ERA5-Land grid cells within each county.

For precipitation, monthly values are first summed to obtain annual precipitation totals for each grid cell. These annual totals are then spatially averaged across the grid cells within each county.

Therefore, `precipitatii_anuale_mm` represents the county-level spatial mean of annual total precipitation.

#### Validation Checks

In [62]:
df_clima.isna().sum()

judet                         0
NUTS_ID                       0
an                            0
temperatura_medie_anuala_C    0
precipitatii_anuale_mm        0
dtype: int64

In [63]:
df_clima.describe()

,an,temperatura_medie_anuala_C,precipitatii_anuale_mm
count,574.000000,574.000000,574.000000
mean,2017.500000,10.559310,716.754231
std,4.034645,1.937717,196.704739
min,2011.000000,6.255729,251.268285
25%,2014.000000,9.067290,590.789059
50%,2017.500000,10.524341,715.409783
75%,2021.000000,12.194525,855.228000
max,2024.000000,14.900117,1260.069888


In [64]:
df_clima.groupby("judet")["an"].nunique().sort_values()

judet
Alba               14
Arad               14
Argeş              14
Bacău              14
Bihor              14
Bistriţa-Năsăud    14
Botoşani           14
Braşov             14
Brăila             14
Buzău              14
Caraş-Severin      14
Cluj               14
Constanţa          14
Covasna            14
Călăraşi           14
Dolj               14
Dâmboviţa          14
Galaţi             14
Giurgiu            14
Gorj               14
Harghita           14
Hunedoara          14
Ialomiţa           14
Iaşi               14
Ilfov              14
Maramureş          14
Mehedinţi          14
Mureş              14
Neamţ              14
Olt                14
Prahova            14
Satu Mare          14
Sibiu              14
Suceava            14
Sălaj              14
Teleorman          14
Timiş              14
Tulcea             14
Vaslui             14
Vrancea            14
Vâlcea             14
Name: an, dtype: int64

#### County Name Standardization

County names are standardized by removing diacritics to ensure consistent identifiers and facilitate subsequent joins with INS datasets.

In [65]:
import unicodedata

def remove_diacritics(text):
    if text is None:
        return text
    text = str(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text

df_clima["judet_std"] = (
    df_clima["judet"]
    .apply(remove_diacritics)
    .str.strip()
    .str.title()
)

In [66]:

df_clima[["judet", "judet_std"]].drop_duplicates().sort_values("judet_std")

,judet,judet_std
23,Alba,Alba
31,Arad,Arad
20,Argeş,Arges
16,Bacău,Bacau
17,Bihor,Bihor
11,Bistriţa-Năsăud,Bistrita-Nasaud
18,Botoşani,Botosani
3,Brăila,Braila
0,Braşov,Brasov
29,Buzău,Buzau


#### Annual Change Calculation

Year-to-year changes in the county-level climate indicators are calculated for subsequent analysis.

In [67]:
df_clima = df_clima.sort_values(["judet", "an"]).copy()

df_clima["schimbare_temp_anuala_C"] = (
    df_clima.groupby("judet")["temperatura_medie_anuala_C"].diff()
)

df_clima["schimbare_precipitatii_anuala_mm"] = (
    df_clima.groupby("judet")["precipitatii_anuale_mm"].diff()
)

df_clima["schimbare_precipitatii_anuala_pct"] = (
    df_clima.groupby("judet")["precipitatii_anuale_mm"].pct_change() * 100
)

df_clima.head()

,judet,NUTS_ID,an,temperatura_medie_anuala_C,precipitatii_anuale_mm,judet_std,schimbare_temp_anuala_C,schimbare_precipitatii_anuala_mm,schimbare_precipitatii_anuala_pct
23,Alba,RO121,2011,8.299947,698.691545,Alba,NaN,NaN,NaN
64,Alba,RO121,2012,8.933329,763.673799,Alba,0.633381,64.982254,9.300564
105,Alba,RO121,2013,8.863476,960.123806,Alba,-0.069853,196.450008,25.724335
146,Alba,RO121,2014,9.634226,875.316853,Alba,0.770750,-84.806953,-8.832918
187,Alba,RO121,2015,9.352633,889.736683,Alba,-0.281593,14.419830,1.647384


#### Final Analysis Period

The year 2011 is excluded after calculating annual changes, resulting in the final analysis period of 2012–2024.

In [69]:
df_clima = df_clima[df_clima["an"] >= 2012].copy()


#### Final Validation Checks

In [70]:
df_clima.head(10)

,judet,NUTS_ID,an,temperatura_medie_anuala_C,precipitatii_anuale_mm,judet_std,schimbare_temp_anuala_C,schimbare_precipitatii_anuala_mm,schimbare_precipitatii_anuala_pct
64,Alba,RO121,2012,8.933329,763.673799,Alba,0.633381,64.982254,9.300564
105,Alba,RO121,2013,8.863476,960.123806,Alba,-0.069853,196.450008,25.724335
146,Alba,RO121,2014,9.634226,875.316853,Alba,0.770750,-84.806953,-8.832918
187,Alba,RO121,2015,9.352633,889.736683,Alba,-0.281593,14.419830,1.647384
228,Alba,RO121,2016,8.720793,1121.871581,Alba,-0.631840,232.134898,26.090292
269,Alba,RO121,2017,8.747192,816.429418,Alba,0.026399,-305.442163,-27.226125
310,Alba,RO121,2018,9.483492,1033.067861,Alba,0.736300,216.638442,26.534865
351,Alba,RO121,2019,9.709572,845.223926,Alba,0.226080,-187.843935,-18.183117
392,Alba,RO121,2020,9.267864,980.267029,Alba,-0.441708,135.043103,15.977198
433,Alba,RO121,2021,8.353869,966.860869,Alba,-0.913995,-13.406160,-1.367603


#### Export Processed Climate Data

In [71]:
from pathlib import Path

# Output directory for processed climate data
output_dir = Path("../../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Export final county-year climate dataset
output_csv = output_dir / "county_year_climate_2012_2024.csv"

df_clima.to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV exported to: {output_csv}")

CSV exported to: ..\..\data\processed\county_year_climate_2012_2024.csv
